## Asset Cashflows Construction

In [1]:
from google.cloud import bigquery
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ID = "insurance-backed-securities"   # <-- fill in
DATASET    = "Securities"      # <-- fill in

client = bigquery.Client(project=PROJECT_ID)
print(f"Connected to project: {client.project}")

/opt/anaconda3/lib/python3.12/site-packages/google/auth/_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


Connected to project: insurance-backed-securities


In [2]:
tables = list(client.list_tables(f"{PROJECT_ID}.{DATASET}"))
for t in tables:
    print(t.table_id)

Agg_Fixed_Field
Agg_Spread_Long


In [5]:
import numpy as np
import pandas_market_calendars as mcal

OUTPUT_TABLE = "Asset_Cashflows"
START_DATE = pd.Timestamp("2024-03-01")
FACE = 100.0

sql = f"""
SELECT CUSIP, Cpn, CPN_FREQ, Maturity
FROM `{PROJECT_ID}.{DATASET}.Agg_Fixed_Field`
WHERE CUSIP IS NOT NULL
  AND Maturity IS NOT NULL
  AND Cpn IS NOT NULL
  AND CPN_FREQ IS NOT NULL
  AND Maturity > DATE('2024-03-01')
"""
bonds_df = client.query(sql).to_dataframe()
bonds_df["Maturity"] = pd.to_datetime(bonds_df["Maturity"])
bonds_df["CPN_FREQ"] = bonds_df["CPN_FREQ"].astype(int)
print(bonds_df.shape)
bonds_df.head()

/opt/anaconda3/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


(303, 4)


,CUSIP,Cpn,CPN_FREQ,Maturity
0,58769JAQ0,4.80,2,2027-01-11
1,YT2012888,5.10,2,2029-11-15
2,05565ECH6,4.90,2,2027-04-02
3,233835AQ0,8.50,2,2031-01-18
4,58769JAC1,5.25,2,2027-11-29


In [6]:
# SIFMA US bond-market holiday calendar (covers weekends + US bond holidays)
cal = mcal.get_calendar("SIFMA_US")
holiday_set = set(pd.to_datetime(cal.holidays().holidays).normalize())

def roll_forward(d):
    """Following Business Day convention: roll to next non-weekend, non-holiday."""
    d = pd.Timestamp(d).normalize()
    while d.weekday() >= 5 or d in holiday_set:
        d += pd.Timedelta(days=1)
    return d

def generate_coupon_dates(maturity, freq, start):
    """Coupon dates anchored to maturity, working backwards in 12/freq-month steps,
    keeping unadjusted dates >= start."""
    interval_months = 12 // freq
    dates = []
    d = pd.Timestamp(maturity)
    while d >= pd.Timestamp(start):
        dates.append(d)
        d = d - pd.DateOffset(months=interval_months)
    dates.sort()
    return dates

In [7]:
rows = []
for r in bonds_df.itertuples(index=False):
    cpn, freq, mat = r.Cpn, r.CPN_FREQ, r.Maturity
    coupon_amt = FACE * cpn / 100.0 / freq
    cpn_dates = generate_coupon_dates(mat, freq, START_DATE)
    for d in cpn_dates:
        rows.append((roll_forward(d), r.CUSIP, coupon_amt, "COUPON"))
    rows.append((roll_forward(mat), r.CUSIP, FACE, "PRINCIPAL"))

cashflows_df = pd.DataFrame(rows, columns=["PaymentDate", "CUSIP", "Payment", "Type"])
cashflows_df["PaymentDate"] = pd.to_datetime(cashflows_df["PaymentDate"]).dt.date
cashflows_df = cashflows_df.sort_values(["CUSIP", "PaymentDate"]).reset_index(drop=True)

print(f"Total cashflow rows: {len(cashflows_df):,}")
print(f"Unique bonds: {cashflows_df['CUSIP'].nunique()}")
print(cashflows_df.head(10))
sample = cashflows_df["CUSIP"].iloc[0]
print(f"\nSample for {sample}:")
print(cashflows_df[cashflows_df["CUSIP"] == sample])

Total cashflow rows: 3,043
Unique bonds: 303
  PaymentDate      CUSIP  Payment    Type
0  2024-08-15  0010EPAF5    3.325  COUPON
1  2025-02-18  0010EPAF5    3.325  COUPON
2  2025-08-15  0010EPAF5    3.325  COUPON
3  2026-02-17  0010EPAF5    3.325  COUPON
4  2026-08-17  0010EPAF5    3.325  COUPON
5  2027-02-16  0010EPAF5    3.325  COUPON
6  2027-08-16  0010EPAF5    3.325  COUPON
7  2028-02-15  0010EPAF5    3.325  COUPON
8  2028-08-15  0010EPAF5    3.325  COUPON
9  2029-02-15  0010EPAF5    3.325  COUPON

Sample for 0010EPAF5:
   PaymentDate      CUSIP  Payment       Type
0   2024-08-15  0010EPAF5    3.325     COUPON
1   2025-02-18  0010EPAF5    3.325     COUPON
2   2025-08-15  0010EPAF5    3.325     COUPON
3   2026-02-17  0010EPAF5    3.325     COUPON
4   2026-08-17  0010EPAF5    3.325     COUPON
5   2027-02-16  0010EPAF5    3.325     COUPON
6   2027-08-16  0010EPAF5    3.325     COUPON
7   2028-02-15  0010EPAF5    3.325     COUPON
8   2028-08-15  0010EPAF5    3.325     COUPON
9   2029-0

In [8]:
table_id = f"{PROJECT_ID}.{DATASET}.{OUTPUT_TABLE}"
job_config = bigquery.LoadJobConfig(
    schema=[
        bigquery.SchemaField("PaymentDate", "DATE"),
        bigquery.SchemaField("CUSIP", "STRING"),
        bigquery.SchemaField("Payment", "FLOAT64"),
        bigquery.SchemaField("Type", "STRING"),
    ],
    write_disposition="WRITE_TRUNCATE",
    time_partitioning=bigquery.TimePartitioning(field="PaymentDate"),
    clustering_fields=["CUSIP"],
)
job = client.load_table_from_dataframe(cashflows_df, table_id, job_config=job_config)
job.result()
print(f"Loaded {job.output_rows:,} rows into {table_id}")

Loaded 3,043 rows into insurance-backed-securities.Securities.Asset_Cashflows


In [9]:
verify_sql = f"""
SELECT CUSIP,
       COUNT(*) AS n_payments,
       MIN(PaymentDate) AS first_pmt,
       MAX(PaymentDate) AS last_pmt,
       ROUND(SUM(Payment), 2) AS total_payment
FROM `{PROJECT_ID}.{DATASET}.{OUTPUT_TABLE}`
GROUP BY CUSIP
ORDER BY CUSIP
LIMIT 10
"""
client.query(verify_sql).to_dataframe()

/opt/anaconda3/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,CUSIP,n_payments,first_pmt,last_pmt,total_payment
0,0010EPAF5,19,2024-08-15,2033-02-15,159.85
1,00138EAY0,6,2024-06-24,2026-06-24,113.37
2,00138EBB9,8,2024-08-20,2027-08-20,116.28
3,00182FBJ4,9,2024-07-22,2028-01-21,113.80
4,00182FBM7,13,2024-08-13,2030-02-13,115.30
5,00388WAL5,11,2024-07-24,2029-01-24,121.88
6,00440FAA2,14,2024-04-01,2030-04-01,163.05
7,022249AU0,9,2024-07-15,2028-01-18,127.00
8,02665WFE6,12,2024-03-13,2029-03-13,126.95
9,04010LBE2,7,2024-07-15,2027-01-15,121.00
